# Random Forest classification development

This notebook uses the **same shared rules and the same split as `sets.ipynb`**: both import `aqi_labels.py`, so the breakpoints, the validity gates, the category boundaries and the train/test partition are identical by construction rather than by two hand-copied implementations kept in step. The source is the 418,753-row cleaned `all_regions` table — the raw 12-station concatenation is 420,768 rows and cleaning removes 2,015 of them.

The daily label lookup has about 17,000 unique station-day rows; that is **not** the model dataset size. Each daily label is attached to its matching original hourly rows. Models train on the hourly training rows and evaluate on the hourly test rows, before rows without a valid daily label are excluded. Pollutant concentrations, pollutant averages, IAQIs, AQI, and label-derived fields are excluded from predictors to avoid direct target leakage. Predictors are hourly meteorology, station, and calendar/time context. Missing numeric predictors are median-imputed; categorical predictors are filled and one-hot encoded. Random Forest does not require feature scaling.

**Split:** train and test are separated by **station-day**, not by hourly row. No station-day appears in both sets, so no test row is scored against a label its own training rows had already seen. All 12 stations appear in both sets, so these metrics describe accuracy on held-out **days at known monitoring stations** — not at stations the model has never seen. A station-holdout evaluation would answer that second question and is not attempted here.

**Label provenance caveat:** the targets count hourly values that `temp_table.ipynb` filled by linear interpolation (at most 3 consecutive hours within a station, recorded in its `interpolated_columns` column) as valid readings, because interpolated hours satisfy the ≥20-hour and ≥14-window gates exactly like measured ones. `sets.ipynb` quantifies how much of each daily value rests on interpolation and reruns the labels from measured hours only as a sensitivity check — see its "Interpolation provenance and measured-only sensitivity" section before quoting any metric here. Interpolation was applied to meteorological columns as well, so a small share of the predictors carries the same provenance.

## 1. Load and confirm the modeling inputs

Load the same `all_regions` MySQL table as `sets.ipynb`, and import the shared `aqi_labels.py` rules. The source row count is checked. The split sizes are deliberately **not** pinned: this notebook calls the same split function as `sets.ipynb`, so they cannot disagree.

In [ ]:
import os
import re
import sys
from pathlib import Path

import numpy as np
import pandas as pd

from IPython.display import display
from sqlalchemy import create_engine, text

try:
    from sklearn.compose import ColumnTransformer
    from sklearn.impute import SimpleImputer
    from sklearn.ensemble import RandomForestClassifier
    from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, balanced_accuracy_score
    from sklearn.pipeline import Pipeline
    from sklearn.preprocessing import OneHotEncoder
except ImportError as error:
    raise ImportError(
        "This modeling notebook requires scikit-learn. Install it in this notebook's "
        "Python environment with `%pip install scikit-learn`, then restart the kernel."
    ) from error

# aqi_labels.py holds the China AQI rules and the train/test split, shared with sets.ipynb
# so the two notebooks cannot drift apart.
NOTEBOOK_DIR = Path.cwd()
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

from aqi_labels import (  # noqa: E402  (import must follow the path bootstrap)
    POLLUTANTS,
    TARGET_COLUMNS,
    assert_disjoint_station_days,
    attach_daily_labels,
    build_daily_labels,
    require_columns,
    split_by_station_day,
)

RANDOM_STATE = 42
TEST_FRACTION = 0.20
SOURCE_TABLE = "all_regions"
# Cleaned all_regions row count: the raw 12-station concatenation is 420,768 rows and
# cleaning removes 2,015 of them. The split sizes are deliberately not pinned here --
# this notebook calls the same split function as sets.ipynb, so they cannot disagree.
EXPECTED_SOURCE_ROWS = 418_753
DB_URL = os.environ.get("AIR_QUALITY_DB_URL", "mysql+pymysql://root:@localhost/air_quality")

if not re.fullmatch(r"[A-Za-z0-9_]+", SOURCE_TABLE):
    raise ValueError(f"Invalid MySQL table name: {SOURCE_TABLE!r}")
engine = create_engine(DB_URL, pool_pre_ping=True)
try:
    with engine.connect() as connection:
        source_df = pd.read_sql_query(text(f"SELECT * FROM {SOURCE_TABLE}"), connection)
except Exception as error:
    raise RuntimeError(
        f"Could not load {SOURCE_TABLE!r} from the air_quality database. Check MySQL, "
        "AIR_QUALITY_DB_URL, the table name, SQLAlchemy, and PyMySQL. "
        f"Original database error: {error}"
    ) from error

print(f"Loaded source rows: {len(source_df):,}")
if len(source_df) != EXPECTED_SOURCE_ROWS:
    raise ValueError(
        f"Expected {EXPECTED_SOURCE_ROWS:,} rows in air_quality.{SOURCE_TABLE}, "
        f"but found {len(source_df):,}. The source must be the cleaned all_regions table."
    )
require_columns(source_df, f"air_quality.{SOURCE_TABLE}")
display(source_df.head())
display(pd.DataFrame({
    "dtype": source_df.dtypes.astype(str),
    "missing_rows": source_df.isna().sum(),
    "missing_percent": (source_df.isna().mean() * 100).round(2),
}))

## 2. Calculate daily China AQI and pollutant category targets

This calls the shared `build_daily_labels` in `aqi_labels.py`, so the 24-hour gate (at least 20 valid hourly readings), the ozone rule (the maximum qualifying trailing 8-hour mean, with the 1-hour breakpoint rule above 800 µg/m³), and the category boundaries are identical to `sets.ipynb` by construction. The daily label table is a station-day lookup only (about 17,000 rows). Each label is joined back to matching hourly rows, so the classifiers use the hourly split—not a 17,000-row model dataset. Invalid or unavailable daily targets remain missing and are excluded from that target's model fit.

In [ ]:
# Daily China AQI targets come from aqi_labels.py, the same function sets.ipynb uses, so
# the breakpoints, the validity gates and the category boundaries are identical by
# construction rather than by inspection.
#
# The result is a station-day lookup table, not the model dataset. Each label is joined
# back to its matching hourly rows in the next cell.
daily_labels, _ = build_daily_labels(source_df)

print(f"Daily station records: {len(daily_labels):,}")
display(daily_labels[TARGET_COLUMNS].notna().sum().rename("valid_station_days").to_frame())

## 3. Recreate the exact hourly training and test sets

This calls the same `split_by_station_day` function as `sets.ipynb` with `RANDOM_STATE = 42`, so the two notebooks produce identical partitions by construction instead of two hand-copied implementations that had to be kept in step. The split is checked to partition the source rows exactly and to share no station-day between the sets. The station-day target labels are then joined to every hourly row using station and date. Original hourly split sizes are printed before rows with missing labels are excluded from individual model fits.

In [ ]:
# The same split function as sets.ipynb, so the two notebooks cannot disagree: 80/20
# within each station, sampling whole station-days so no station-day spans the boundary.
train_raw, test_raw = split_by_station_day(
    source_df, test_fraction=TEST_FRACTION, random_state=RANDOM_STATE
)
train_days, test_days = assert_disjoint_station_days(train_raw, test_raw)

print(f"Complete source rows: {len(source_df):,}")
print(f"Training set: {len(train_raw):,} rows")
print(f"Test set: {len(test_raw):,} rows")
print(f"Total after split: {len(train_raw) + len(test_raw):,} rows")
print(f"Station-days: {train_days:,} training / {test_days:,} test, no station-day in both")

# Broadcast each station-day target onto that station's hourly rows.
train_set = attach_daily_labels(train_raw, daily_labels)
test_set = attach_daily_labels(test_raw, daily_labels)

print("Hourly model input rows (before excluding rows without a valid target):")
print(f"Training partition: {len(train_set):,}")
print(f"Test partition: {len(test_set):,}")
print("Unique daily label rows are looked up and broadcast to these hourly rows.")

# Define hourly predictors explicitly; pollutant values and all AQI/target fields stay out.
WEATHER_FEATURES = [column for column in ["TEMP", "PRES", "DEWP", "RAIN", "WSPM"] if column in source_df.columns]
if not WEATHER_FEATURES:
    raise KeyError("No expected weather predictors were found (TEMP, PRES, DEWP, RAIN, WSPM).")
for frame in (train_set, test_set):
    frame["year"] = pd.to_numeric(frame["year"], errors="coerce")
    frame["month"] = pd.to_numeric(frame["month"], errors="coerce")
    frame["hour"] = pd.to_numeric(frame["hour"], errors="coerce")
    row_time = frame[["year", "month", "day", "hour"]].apply(pd.to_numeric, errors="coerce")
    day_of_year = pd.to_datetime({
        "year": row_time["year"], "month": row_time["month"],
        "day": row_time["day"], "hour": row_time["hour"],
    }, errors="coerce").dt.dayofyear
    frame["month_sin"] = np.sin(2 * np.pi * (frame["month"] - 1) / 12)
    frame["month_cos"] = np.cos(2 * np.pi * (frame["month"] - 1) / 12)
    frame["day_of_year_sin"] = np.sin(2 * np.pi * (day_of_year - 1) / 365.25)
    frame["day_of_year_cos"] = np.cos(2 * np.pi * (day_of_year - 1) / 365.25)
    frame["hour_sin"] = np.sin(2 * np.pi * frame["hour"] / 24)
    frame["hour_cos"] = np.cos(2 * np.pi * frame["hour"] / 24)

NUMERIC_FEATURES = [
    *WEATHER_FEATURES, "year", "month_sin", "month_cos",
    "day_of_year_sin", "day_of_year_cos", "hour_sin", "hour_cos",
]
CATEGORICAL_FEATURES = ["station"] + (["wd"] if "wd" in source_df.columns else [])
FEATURE_COLUMNS = [*NUMERIC_FEATURES, *CATEGORICAL_FEATURES]
print("Predictor columns:", FEATURE_COLUMNS)
print("Target columns:", TARGET_COLUMNS)
print("Training rows with valid targets:")
display(train_set[TARGET_COLUMNS].notna().sum().rename("rows").to_frame())
print("Test rows with valid targets:")
display(test_set[TARGET_COLUMNS].notna().sum().rename("rows").to_frame())

## 4. Train and evaluate the Random Forest classifiers

One classifier is fitted for the overall AQI category and one for each pollutant category. The models use the exact hourly training partition above. For each target, rows without that target label are removed from that model’s fit/evaluation only; the original split remains unchanged. Preprocessing is fit on training rows only.


In [ ]:
models = {}
evaluation_rows = []

for target in TARGET_COLUMNS:
    train_task = train_set.dropna(subset=[target]).copy()
    test_task = test_set.dropna(subset=[target]).copy()
    if train_task.empty or test_task.empty:
        print(f"Skipping {target}: no labeled rows in train or test partition.")
        continue
    y_train = train_task[target].astype(str)
    y_test = test_task[target].astype(str)
    if y_train.nunique() < 2:
        print(f"Skipping {target}: training partition has fewer than two classes.")
        continue
    X_train = train_task[FEATURE_COLUMNS].copy()
    X_test = test_task[FEATURE_COLUMNS].copy()

    numeric_pipeline = Pipeline([("imputer", SimpleImputer(strategy="median"))])
    categorical_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="constant", fill_value="Unknown")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ])
    preprocessing = ColumnTransformer([
        ("numeric", numeric_pipeline, NUMERIC_FEATURES),
        ("categorical", categorical_pipeline, CATEGORICAL_FEATURES),
    ])
    model = Pipeline([
        ("preprocessing", preprocessing),
        ("classifier", RandomForestClassifier(
            n_estimators=300,
            min_samples_leaf=2,
            class_weight="balanced_subsample",
            random_state=RANDOM_STATE,
            n_jobs=-1,
        )),
    ])
    model.fit(X_train, y_train)
    predictions = model.predict(X_test)
    models[target] = model

    labels = sorted(set(y_train) | set(y_test))
    print(f"\n=== {target} ===")
    print(f"Train rows: {len(train_task):,} | Test rows: {len(test_task):,}")
    print(f"Accuracy: {accuracy_score(y_test, predictions):.3f}")
    print(f"Balanced accuracy: {balanced_accuracy_score(y_test, predictions):.3f}")
    display(pd.DataFrame(
        confusion_matrix(y_test, predictions, labels=labels),
        index=pd.Index(labels, name="actual"),
        columns=pd.Index(labels, name="predicted"),
    ))
    display(pd.DataFrame(classification_report(
        y_test, predictions, labels=labels, output_dict=True, zero_division=0
    )).T)
    feature_names = model.named_steps["preprocessing"].get_feature_names_out()
    importances = model.named_steps["classifier"].feature_importances_
    display(pd.Series(importances, index=feature_names).sort_values(ascending=False).head(10).rename("importance").to_frame())
    evaluation_rows.append({
        "target": target,
        "train_rows": len(train_task),
        "test_rows": len(test_task),
        "accuracy": accuracy_score(y_test, predictions),
        "balanced_accuracy": balanced_accuracy_score(y_test, predictions),
    })

model_results = pd.DataFrame(evaluation_rows).set_index("target")
display(model_results)
